In [ ]:
from mipt import *
import numpy as np
from numpy.typing import NDArray
from scipy.optimize import curve_fit

Array = NDArray[np.complex128]

def ghz_white_noise_state(parties: int, p_noise: float) -> np.ndarray:
    """
    rho(p) = (1 - p) |GHZ_n><GHZ_n| + p I / 2**n
    """
    d = 2 ** parties

    psi = np.zeros(d, dtype=np.complex128)
    psi[0] = 1.0 / np.sqrt(2.0)
    psi[-1] = 1.0 / np.sqrt(2.0)

    rho_ghz = np.outer(psi, psi.conj())

    return (1.0 - p_noise) * rho_ghz + p_noise * np.eye(d) / d

ps = np.linspace(0, 1, 15)
gmn_values = []
for p in ps:
    rho = ghz_white_noise_state(parties=4, p_noise=p)
    score = gmn(
            rho,
            parties=4,
            formulation="monotone",
            return_problem=False,
        )
    gmn_values.append(score)

plt.plot(ps, gmn_values, '.')

In [ ]:
from mipt import *
import csv

n = 8
d = 2*n
ps = np.linspace(0.0, 1.0, 11)
realisations = 100

with open("gmn.csv", mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["p", "gmn"])

means = []
stderrs = []
for p in ps:
    print(f"Simulating p = {p}...")
    gmns = []
    for r in range(realisations):
        rhos = get_mipt_rho_1d(n, d, p, subsyst=3, all_matrices=True, closed=True)
        this_gmns = [gmn(rho, parties=3) for rho in rhos]
        gmns.extend(this_gmns)
        
        new_rows = [[p, g] for g in this_gmns]
        with open("gmn.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            for row in new_rows:
                writer.writerow(row)

        print(f"Realisations: {r}/{realisations}", end='\r', flush=True)
    
    means.append(np.mean(gmns))
    stderrs.append(np.std(gmns)/np.sqrt(realisations*n))
    print(f"\nGMN: {means[-1]:.8g} ± {stderrs[-1]:.3g}")


fig, ax1 = plt.subplots(1, 1, figsize=(10, 4))
ax1.errorbar(ps_unique, [1 - m for m in means], yerr=stderrs, fmt='.', color='black', ecolor='black', capsize=3, elinewidth=1)
ax1.set_xlabel('Measurement rate $p$')
ax1.set_ylabel(r'GMN $\tilde{N}_g(\rho)$ = $-\min\;\text{tr}\.(\rho W)$')

props = dict(boxstyle='square', facecolor='white', alpha=0.5)
ax1.text(0.75, 0.95, f"n = {n}\nPeriods = {int(d/2)}\nRealisations = {realisations}", transform=ax1.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)
 

fig.tight_layout()


In [ ]:
from mipt import *

def max_ent_gate():
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    return qc
    

def max_ent_mipt_1d(n, d, p, closed=True):
    """n must be even. p must be between 0 and 1."""
    if n % 2 != 0:
        raise ValueError("n must be even")
    if not (0 <= p <= 1):
        raise ValueError("p must be between 0 and 1")
    
    qc = QuantumCircuit(n, n)
    U = max_ent_gate()
    for layer in range(d):

        if (layer % 2 == 0):
            # even layer
            for qubit in range(0, n, 2):
                qc.append(U, [qubit, (qubit + 1)])
        else:
            # odd layer
            if closed:
                qc.append(U, [0, -1])
            for qubit in range(1, n - 1, 2):
                qc.append(U, [qubit, (qubit + 1)])
        
        # random measurements with probability p
        for qubit in range(n):
            if random.random() < p:
                qc.measure(qubit, qubit)

    if (random.random() < 0.5): 
    # 50% chance of an extra even layer at the end
        for qubit in range(0, n, 2):
            qc.append(U, [qubit, (qubit + 1)])
        for qubit in range(n):
            if random.random() < p:
                qc.measure(qubit, qubit)

    return qc

def get_max_ent_mipt_rho_1d(n, d, p, subsyst=3, all_matrices=False, closed=True):

    subsystems = []
    if all_matrices and closed:
        subsystems = [[ (i + j) % n for j in range(subsyst) ] for i in range(n)]
    else:
        subsystems = [list(range(subsyst))]

    backend = AerSimulator(device="CPU", method="matrix_product_state")
    qc = max_ent_mipt_1d(n=n, d=d, p=float(p), closed=closed)

    for i in range(len(subsystems)):
        qc.save_density_matrix(
            qubits=subsystems[i],
            label=f"final_state_{i}",
            pershot=True,
        )

    tqc = transpile(qc, backend)

    # One conditional outcome trajectory for this independently drawn circuit.
    result = backend.run(tqc, shots=1).result()

    if len(subsystems) == 1:
        return result.data(0)["final_state_0"][0]

    data = []
    for i in range(len(subsystems)):
        data.append(result.data(0)[f"final_state_{i}"][0])

    return data

n = 8
d = 2*n
ps = np.linspace(0.0, 1.0, 11)
reps = 5

# means, stderrs = [], []
gmns = []

for p in ps:
    print(f"p = {p}:")
    # for _ in reps:
    rhos = get_max_ent_mipt_rho_1d(n, d, p, all_matrices=True)
    print(rhos)
    print([gmn(rho, parties=3) for rho in rhos])

# print(gmns)





In [ ]:
from numpy.typing import NDArray
ComplexVector = NDArray[np.complex128]
ComplexMatrix = NDArray[np.complex128]
from gnme import inflation_score
import numpy as np
from read_mipt_rho3 import read_rho3_bin
from collections import defaultdict

def scan_func_by_p(records, func, limit=None):
    """
    Compute mean ± sample stddev of a given function at each p.

    Parameters
    ----------
    records : list[dict]
        Output from read_rho3_bin(...).

    func : callable
        Function taking one 8x8 density matrix and returning a scalar.

    limit : int | None
        Maximum number of matrices to evaluate for each p.
        If None, all matrices are used.

    Returns
    -------
    results : dict[float, dict]
    """

    if limit is not None:
        if not isinstance(limit, int):
            raise TypeError("limit must be an int or None.")
        if limit <= 0:
            raise ValueError("limit must be positive when provided.")

    grouped = defaultdict(list)

    for rec in records:
        grouped[float(rec["p"])].append(rec)

    results = {}

    for p in sorted(grouped):
        group = grouped[p]

        if limit is not None:
            group = group[:limit]

        values = np.asarray(
            [
                float(np.real_if_close(func(rec["rho"])))
                for rec in group
            ],
            dtype=float,
        )

        results[p] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)) if len(values) > 1 else 0.0,
            "n": int(len(values)),
            "values": values,
        }

    return results

def mixed_ghz_3q(p):
    # Define the basis states
    ket0 = np.array([1, 0])
    ket1 = np.array([0, 1])

    # |000> state = |0> ⊗ |0> ⊗ |0>
    ket000 = np.kron(np.kron(ket0, ket0), ket0)

    # |111> state = |1> ⊗ |1> ⊗ |1>
    ket111 = np.kron(np.kron(ket1, ket1), ket1)

    # 1/sqrt(2) * (|000> + |111>)
    ghz_state = (1 / np.sqrt(2)) * (ket000 + ket111)

    # Density matrix representation: rho = |GHZ><GHZ|
    return (1-p)*np.outer(ghz_state, ghz_state) + p * np.eye(8) / 8


# records = read_rho3_bin("rho3.bin")

# results = scan_func_by_p(records, inflation_score, limit=10)

# for p, stats in results.items():
#     print(
#         f"p={p:.3f}: "
#         f"F_GHZ = {stats['mean']:.6f} ± {stats['std']:.6f} "
#         f"(n={stats['n']})"
#     )

# p_vals = np.array(sorted(results.keys()))
# means = np.array([results[p]["mean"] for p in p_vals])
# stds = np.array([results[p]["std"] for p in p_vals])
# ns = np.array([results[p]["n"] for p in p_vals])

def ket0():
    return np.array([1, 0], dtype=np.complex128)


def ket1():
    return np.array([0, 1], dtype=np.complex128)


def dm(psi):
    psi = np.asarray(psi, dtype=np.complex128)
    return np.outer(psi, psi.conj())


def normalize(psi):
    psi = np.asarray(psi, dtype=np.complex128)
    return psi / np.linalg.norm(psi)


def kron_all(*ops):
    out = np.array([[1]], dtype=np.complex128)
    for op in ops:
        out = np.kron(out, op)
    return out


def pure_product_abc(a, b, c):
    psi = np.kron(np.kron(a, b), c)
    return dm(psi)


def bell_phi_plus():
    return normalize(np.array([1, 0, 0, 1], dtype=np.complex128))


def ghz3():
    psi = np.zeros(8, dtype=np.complex128)
    psi[0] = 1 / np.sqrt(2)
    psi[7] = 1 / np.sqrt(2)
    return dm(psi)


def w3():
    psi = np.zeros(8, dtype=np.complex128)
    psi[1] = 1 / np.sqrt(3)  # |001>
    psi[2] = 1 / np.sqrt(3)  # |010>
    psi[4] = 1 / np.sqrt(3)  # |100>
    return dm(psi)


def white_mix(rho, p):
    """
    rho_p = (1-p) rho + p I/d
    where p is the white-noise fraction.
    """
    d = rho.shape[0]
    return (1 - p) * rho + p * np.eye(d, dtype=np.complex128) / d


def bell_ab_product_c():
    """
    |Phi+>_AB tensor |0>_C
    in output order A,B,C.
    """
    bell = bell_phi_plus()  # order A,B
    c = ket0()

    psi = np.kron(bell, c)  # AB,C -> A,B,C
    return dm(psi)


def bell_ac_product_b():
    """
    |Phi+>_AC tensor |0>_B
    output order A,B,C.
    """
    psi = np.zeros(8, dtype=np.complex128)

    # (|00>_AC + |11>_AC)/sqrt(2) tensor |0>_B
    # output order A,B,C:
    # A=0,B=0,C=0 -> |000>
    # A=1,B=0,C=1 -> |101>
    psi[0b000] = 1 / np.sqrt(2)
    psi[0b101] = 1 / np.sqrt(2)

    return dm(psi)


def bell_bc_product_a():
    """
    |0>_A tensor |Phi+>_BC
    output order A,B,C.
    """
    a = ket0()
    bell = bell_phi_plus()  # order B,C

    psi = np.kron(a, bell)  # A,BC -> A,B,C
    return dm(psi)

rho_000 = pure_product_abc(ket0(), ket0(), ket0())
rho_111 = pure_product_abc(ket1(), ket1(), ket1())

print("product |000>:", inflation_score(rho_000, verbose=False))
print("product |111>:", inflation_score(rho_111, verbose=False))

tests = {
    "Bell_AB x |0>_C": bell_ab_product_c(),
    "Bell_AC x |0>_B": bell_ac_product_b(),
    "Bell_BC x |0>_A": bell_bc_product_a(),
}

for name, rho in tests.items():
    val = inflation_score(rho, verbose=False)
    print(f"{name:20s}  t_max = {val:.8f}")

rho_bisep_mix = (
    0.25 * bell_ab_product_c()
    + 0.35 * bell_ac_product_b()
    + 0.40 * bell_bc_product_a()
)

print("mixed biseparable network:", inflation_score(rho_bisep_mix, verbose=False))

In [ ]:
def haar_unitary(dim, rng):
    z = rng.normal(size=(dim, dim)) + 1j * rng.normal(size=(dim, dim))
    q, r = np.linalg.qr(z)

    diag = np.diag(r)
    phases = np.ones(dim, dtype=np.complex128)
    nz = np.abs(diag) > 0
    phases[nz] = diag[nz] / np.abs(diag[nz])

    return q * phases.conj()


def apply_unitary_to_subsystems_statevec(psi, U, targets, dims):
    """
    Apply U to selected subsystems of a pure state vector.

    dims:
        subsystem dimensions.

    targets:
        subsystem indices acted on by U, in the order U expects.
    """
    dims = list(dims)
    n = len(dims)
    targets = list(targets)

    psi_t = psi.reshape(dims)

    perm = targets + [i for i in range(n) if i not in targets]
    inv_perm = np.argsort(perm)

    front_dim = int(np.prod([dims[i] for i in targets]))
    back_dim = int(np.prod([dims[i] for i in range(n) if i not in targets]))

    psi_perm = np.transpose(psi_t, perm).reshape(front_dim, back_dim)
    psi_perm = U @ psi_perm

    psi_out = psi_perm.reshape([dims[i] for i in perm])
    psi_out = np.transpose(psi_out, inv_perm)

    return psi_out.reshape(-1)


def partial_trace_numeric(rho, dims, keep):
    """
    Numeric ordered marginal, keeping subsystems in the listed order.
    """
    dims = list(dims)
    keep = list(keep)
    n = len(dims)

    trace = [i for i in range(n) if i not in keep]

    T = rho.reshape(*dims, *dims)
    current_dims = list(dims)

    for s in sorted(trace, reverse=True):
        T = np.trace(T, axis1=s, axis2=s + len(current_dims))
        current_dims.pop(s)

    remaining = [i for i in range(n) if i in keep]

    perm_ket = [remaining.index(i) for i in keep]
    k = len(keep)
    perm = perm_ket + [p + k for p in perm_ket]

    T = np.transpose(T, perm)

    d_keep = int(np.prod([dims[i] for i in keep]))
    return T.reshape(d_keep, d_keep)


def triangle_bell_network_state(seed=0, keep_local=(0, 0, 0)):
    """
    Build a 3-qubit state known to be in the triangle network set.

    Virtual subsystem order:
        [A0, A1, B0, B1, C0, C1]

    Link Bell pairs:
        A0-B0
        B1-C0
        C1-A1

    Local nodes:
        A = [A0, A1] = [0, 1]
        B = [B0, B1] = [2, 3]
        C = [C0, C1] = [4, 5]

    keep_local:
        Which local qubit to keep at each node after the local unitary.
        For example (0,0,0) keeps A0, B0, C0.
    """
    rng = np.random.default_rng(seed)

    dims = [2, 2, 2, 2, 2, 2]
    d = 64

    # Build the 6-qubit state:
    # Bell(A0,B0) Bell(B1,C0) Bell(C1,A1)
    #
    # We fill amplitudes explicitly in virtual order [A0,A1,B0,B1,C0,C1].
    psi = np.zeros(d, dtype=np.complex128)

    for x in [0, 1]:  # A0 = B0
        for y in [0, 1]:  # B1 = C0
            for z in [0, 1]:  # C1 = A1
                A0 = x
                B0 = x

                B1 = y
                C0 = y

                C1 = z
                A1 = z

                bits = [A0, A1, B0, B1, C0, C1]
                idx = 0
                for bit in bits:
                    idx = 2 * idx + bit

                psi[idx] = 1 / np.sqrt(8)

    # Random local unitaries at nodes.
    UA = haar_unitary(4, rng)
    UB = haar_unitary(4, rng)
    UC = haar_unitary(4, rng)

    psi = apply_unitary_to_subsystems_statevec(psi, UA, [0, 1], dims)
    psi = apply_unitary_to_subsystems_statevec(psi, UB, [2, 3], dims)
    psi = apply_unitary_to_subsystems_statevec(psi, UC, [4, 5], dims)

    rho6 = dm(psi)

    # Choose one output qubit from each node.
    A_keep = [0, 1][keep_local[0]]
    B_keep = [2, 3][keep_local[1]]
    C_keep = [4, 5][keep_local[2]]

    rho_abc = partial_trace_numeric(rho6, dims, [A_keep, B_keep, C_keep])

    # Symmetrize tiny numerical noise.
    rho_abc = (rho_abc + rho_abc.conj().T) / 2
    rho_abc = rho_abc / np.trace(rho_abc)

    return rho_abc

for seed in range(5):
    rho_net = triangle_bell_network_state(seed=seed, keep_local=(0, 0, 0))
    val = inflation_score(rho_net, verbose=False)
    print(f"triangle Bell network seed={seed}: t_max = {val:.8f}")

In [ ]:
import time
import scs
from gnme import RingInflationSDP3Q
from mipt import *

n=4
d=2*n

rho = get_mipt_rho_1d(n, d, p=0.3, subsyst=3, closed=True)

def test_backend(label, solve_kwargs):
    sdp = RingInflationSDP3Q()
    t0 = time.perf_counter()

    try:
        out = sdp.solve(
            rho,
            verbose=True,
            return_info=True,
            max_iters=200, 
            tol=1e-3,
            **solve_kwargs,
        )
        dt = time.perf_counter() - t0
        print("\n", label)
        print("elapsed:", dt)
        print("out:", out)
    except Exception as e:
        dt = time.perf_counter() - t0
        print("\n", label, "FAILED after", dt)
        print(type(e).__name__, e)

test_backend("CPU default", dict(gpu=False))

test_backend("CPU indirect", dict(
    gpu=False,
    use_indirect=True,
))

test_backend("cuDSS", dict(
    gpu=True,
))

In [ ]:
import numpy as np
from qiskit.quantum_info import partial_trace, DensityMatrix
from mipt import *

def dm(psi):
    psi = np.asarray(psi, dtype=np.complex128)
    return np.outer(psi, psi.conj())


def ghz3():
    psi = np.zeros(8, dtype=np.complex128)
    psi[0] = 1 / np.sqrt(2)
    psi[7] = 1 / np.sqrt(2)
    return dm(psi)


def ghz4():
    psi = np.zeros(16, dtype=np.complex128)
    psi[0] = 1 / np.sqrt(2)
    psi[15] = 1 / np.sqrt(2)
    return dm(psi)

n = 4
d = 2*n
p = 0.0

rho = get_mipt_rho_1d(n, d, p, subsyst=3, closed=True)

rho.draw("hinton")
